In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
import sys
sys.path.append(r'C:\Users\julia\OneDrive\Escritorio\Trabajo\building_ml_models_for_protein_science\src')

In [12]:
from building_models.commons_functions.parsers_commons import ParsersCommons
from building_models.utils.constants import COLUMNS_TO_WORK
from building_models.utils.utils_functions import UtilsFunctions
import pandas as pd

In [13]:
path_export = "../../processed_dataset/"
path_input = "../../raw_dataset/"
metadata_file = "../../raw_dataset/raw_data_description.xlsx"
name_task = "antioxidant_classification"
name_source = "Zhai et al"

In [14]:
df_anti = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/anti.txt")
df_anti["label"] = 1
df_anti = df_anti.drop(columns=["id"])
df_anti.head()


,sequence,label
0,MTKGILLGDKFPDFRAETNEGFIPSFYDWIGKDSWAILFSHPRDFT...,1
1,MLPGLALLLLAAWTARALEVPTDGNAGLLAEPQIAMFCGRLNMHMN...,1
2,MAIALSSSSTITSITLQPKLKTIHGLGTVLPGYSVKSHFRSVSLRR...,1
3,MITSSKKIVSAMLSTSLWIGVASAAYAETTNVEAEGYSTIGGTYQD...,1
4,MANSGLWELITIGSAVRNVAKSYLKAEASSITAKQLYDASKITSSK...,1


In [15]:
df_nonanti = ParsersCommons.read_fasta_doc(f"{path_input}/{name_source}/nonanti.txt")
df_nonanti["label"] = 0
df_nonanti = df_nonanti.drop(columns=["id"])

df_nonanti.head()

,sequence,label
0,MERPYACPVESCDRRFSRSADLTRHIRIHTGQKPFQCRICMRNFSR...,0
1,MAPAEILNGKEISAQIRARLKNQVTQLKEQVPGFTPRLAILQVGNR...,0
2,MNLTELKNTPVSELITLGENMGLENLARMRKQDIIFAILKQHAKSG...,0
3,MKNLDCWVDNEEDIDVILKKSTILNLDINNDIISDISGFNSSVITY...,0
4,MFKVYGYDSNIHKCGPCDNAKRLLTVKKQPFEFINIMPEKGVFDDE...,0


- Concat and check duplicates

In [16]:
df_concat = pd.concat([df_anti, df_nonanti], axis=0, ignore_index=True)
df_concat.shape, df_concat["sequence"].unique().shape

((1805, 2), (1805,))

- Reading metadata

In [17]:
metadata_file = ParsersCommons.read_metadata(metadata_file, name_source=name_source, columns_to_select=COLUMNS_TO_WORK)
metadata_file.head()

,name dataset,name source,type source,static-dynamic,license,reports constant updates,year of publication,last update date,download date,file format,protein format,category dataset,task,obtaining negative dataset,obtaining positive dataset,repository or server,publication,unit of measurement
54,anti.txt,Zhai et al,Dataset,Static,Creative Commons Attribution License,No,2020,2020-10-28,2025-09-07,txt,"Sequence, UniProt ID",Enzyme/protein classification,Antioxidant,No information,"Obtained from other databases, Previously publ...",https://github.com/MAX-zyx/antioxidant_dataset,https://www.frontiersin.org/journals/cell-and-...,No information
55,nonanti.txt,Zhai et al,Dataset,Static,Creative Commons Attribution License,No,2020,2020-10-28,2025-09-07,txt,"Sequence, UniProt ID",Enzyme/protein classification,Antioxidant,"Previously published model dataset, Sampling f...",No information,https://github.com/MAX-zyx/antioxidant_dataset,https://www.frontiersin.org/journals/cell-and-...,No information


In [18]:
dict_metadata = ParsersCommons.create_metadata_from_file(metadata_file)
dict_metadata

{'name dataset': 'anti.txt;nonanti.txt',
 'name source': 'Zhai et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution License',
 'reports constant updates': 'No',
 'year of publication': 2020,
 'last update date': Timestamp('2020-10-28 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'txt',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'No information;Previously published model dataset, Sampling from UniProt',
 'obtaining positive dataset': 'Obtained from other databases, Previously published model dataset, Sampling from UniProt;No information',
 'repository or server': 'https://github.com/MAX-zyx/antioxidant_dataset',
 'publication': 'https://www.frontiersin.org/journals/cell-and-developmental-biology/articles/10.3389/fcell.2020.591487/full',
 'unit of measurement': 'No information',
 'number_of_sour

In [19]:
dict_metadata['number_of_records'] = df_concat.shape[0]
dict_metadata['number_of_collected_sequences'] = df_concat.shape[0]
dict_metadata['number_of_unique_sequences'] = df_concat.shape[0]
dict_metadata['positive_examples'] = df_concat[df_concat["label"] == 1].shape[0]
dict_metadata['negative_examples'] = df_concat[df_concat["label"] == 0].shape[0]
dict_metadata['number_of_sequences_with_errors'] = 0
dict_metadata

{'name dataset': 'anti.txt;nonanti.txt',
 'name source': 'Zhai et al',
 'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'Creative Commons Attribution License',
 'reports constant updates': 'No',
 'year of publication': 2020,
 'last update date': Timestamp('2020-10-28 00:00:00'),
 'download date': Timestamp('2025-09-07 00:00:00'),
 'file format': 'txt',
 'protein format': 'Sequence, UniProt ID',
 'category dataset': 'Enzyme/protein classification',
 'task': 'Antioxidant',
 'obtaining negative dataset': 'No information;Previously published model dataset, Sampling from UniProt',
 'obtaining positive dataset': 'Obtained from other databases, Previously published model dataset, Sampling from UniProt;No information',
 'repository or server': 'https://github.com/MAX-zyx/antioxidant_dataset',
 'publication': 'https://www.frontiersin.org/journals/cell-and-developmental-biology/articles/10.3389/fcell.2020.591487/full',
 'unit of measurement': 'No information',
 'number_of_sour

- Export data

In [20]:
UtilsFunctions.make_directory(f"{path_export}{name_task}/{name_source}")
UtilsFunctions.export_json(f"{path_export}{name_task}/{name_source}/metadata_{name_source}.json", dict_metadata)
df_concat.to_csv(f"{path_export}{name_task}/{name_source}/processed_data.csv", index=False)